# PC recurrent population benchmark

This notebook loads the formal Purkinje-cell assembly from `examples.neuron_compare.cell.pc_ma2024.pc_braincell.PC`, builds one homogeneous PC population of size `N`, and connects the population to itself through the `braincell.Network` recurrent path.

The timing table separates Python/build work, cold `Network.run(...)` time, and warm `Network.run(...)` time. Cold run includes tracing/JIT compilation for a new shape; warm run repeats the same shape and synchronizes results before stopping the timer.

In [ ]:
import os

bench_gpu = os.environ.get("BENCH_GPU")
if bench_gpu and "CUDA_VISIBLE_DEVICES" not in os.environ:
    os.environ["CUDA_VISIBLE_DEVICES"] = bench_gpu
os.environ["JAX_PLATFORMS"] = "cuda"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import sys
import time
from pathlib import Path


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "examples").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import brainstate
import brainunit as u
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import braincell
from braincell import mech
from braincell.filter import at

from examples.neuron_compare.cell.pc_ma2024.parameters import (
    DEFAULT_INDIV,
    DEFAULT_MORPH_PATH,
    load_pc24_params,
)
from examples.neuron_compare.cell.pc_ma2024.pc_braincell import PC as BrainCellPC

brainstate.environ.set(precision=64)

print("braincell version:", braincell.__version__)
print("repo_root:", repo_root)
print("morphology:", DEFAULT_MORPH_PATH)
print("PC parameter indiv:", DEFAULT_INDIV)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("XLA_PYTHON_CLIENT_PREALLOCATE:", os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"))

backend = jax.default_backend()
devices = [str(device) for device in jax.devices()]
print("JAX backend:", backend)
print("JAX devices:", devices)
if backend != "gpu":
    raise RuntimeError(f"Expected JAX CUDA backend, got {backend!r}.")

smoke_start = time.perf_counter()
x = jnp.ones((1024, 1024), dtype=jnp.float32)
y = (x @ x).block_until_ready()
smoke_seconds = time.perf_counter() - smoke_start
print("CUDA matmul smoke sum:", float(jnp.sum(y)))
print("CUDA matmul smoke seconds:", smoke_seconds)


In [ ]:
params = load_pc24_params()

# The formal PC morphology/mechanism stack is heavy. Start conservatively;
# increase this list after the local CUDA runtime is confirmed stable.
N_LIST = [10]
ALLOW_SELF = False
CONNECTION_MODE = "probability"  # "probability" or "all_to_all"
CONNECTION_PROBABILITY = 1.0
CONNECTION_SEED = 7

DT = 0.1 * u.ms
DURATION = 20 * u.ms
WARM_REPEATS = 2
SYNAPTIC_WEIGHT = 0.005 * u.uS
SYNAPTIC_DELAY = 2 * u.ms
EVENT_BACKEND = "auto"
BRAINEVENT_BACKEND = "jax_raw"
SPIKE_RECORDING = "population"  # "full", "population", or "none"
ENABLE_JAX_PROFILE = False
PROFILE_DIR = "/tmp/braincell_pc_recurrent_profile"

TEMPERATURE_CELSIUS = 36.0
V_INIT_MV = -65.0
V_THRESHOLD = 0.0 * u.mV

print("N_LIST:", N_LIST)
print("dt:", DT, "duration:", DURATION)
print("warm repeats:", WARM_REPEATS)
print("connection:", CONNECTION_MODE, "p=", CONNECTION_PROBABILITY, "allow_self=", ALLOW_SELF)
print("event_backend:", EVENT_BACKEND, "brainevent_backend:", BRAINEVENT_BACKEND)
print("spike_recording:", SPIKE_RECORDING)
print("jax profile enabled:", ENABLE_JAX_PROFILE, "profile dir:", PROFILE_DIR)


In [ ]:
def pulse_protocol(size: int) -> mech.CurrentClamp:
    base_delays_ms = np.asarray([0.10, 0.15, 0.20, 0.25], dtype=float)
    delays_ms = base_delays_ms[np.arange(size) % len(base_delays_ms)]
    amplitudes = u.Quantity(1.8 + 0.05 * np.arange(size, dtype=float), u.nA)
    return mech.CurrentClamp(
        delay=u.Quantity(delays_ms, u.ms),
        durations=0.25 * u.ms,
        amplitudes=amplitudes,
    )


def build_pc_population(size: int):
    pc = BrainCellPC(
        DEFAULT_MORPH_PATH,
        params=params,
        temperature_celsius=TEMPERATURE_CELSIUS,
        v_init_mV=V_INIT_MV,
        pop_size=(size,),
        name=f"pc_recurrent_{size}",
    ).build()
    cell = pc.cell
    cell.V_th = V_THRESHOLD
    cell.place(at("soma", 0.5), pulse_protocol(size))
    cell.place(at("soma", 0.5), mech.StateProbe(name="v", field="v"))
    cell.place(at("soma", 0.5), mech.MechanismProbe(name="g_syn", mechanism="exp", field="g"))
    cell.place(at("soma", 0.5), mech.CurrentProbe(name="i_syn", mechanism="exp"))
    cell.place(
        at("soma", 0.5),
        mech.Synapse(
            "ExpSyn",
            tau=2.0 * u.ms,
            e=0.0 * u.mV,
            weight=1.0 * u.uS,
            name="exp",
        ),
    )
    return cell


def recurrent_edge_method():
    if CONNECTION_MODE == "all_to_all":
        return braincell.network.all_to_all(allow_self=ALLOW_SELF)
    if CONNECTION_MODE == "probability":
        return braincell.network.probability(
            p=CONNECTION_PROBABILITY,
            seed=CONNECTION_SEED,
            allow_self=ALLOW_SELF,
        )
    raise ValueError(f"Unknown CONNECTION_MODE={CONNECTION_MODE!r}.")


def add_recurrent_projection(net, edges):
    if edges.n_edge == 0:
        return None
    weights = np.full(edges.n_edge, SYNAPTIC_WEIGHT.to_decimal(u.uS), dtype=float) * u.uS
    return net.add_projection(
        name="PC_to_PC_exp",
        edges="PC_to_PC",
        synapse="exp",
        weight=weights,
        delay=SYNAPTIC_DELAY,
    )


def array_2d(value):
    arr = np.asarray(value, dtype=float)
    return arr.reshape((arr.shape[0], -1))


def trace_2d(value, unit):
    arr = np.asarray(value.to_decimal(unit), dtype=float)
    return arr.reshape((arr.shape[0], -1))


def block_until_ready_tree(value):
    if hasattr(value, "block_until_ready"):
        value.block_until_ready()
        return
    if isinstance(value, dict):
        for item in value.values():
            block_until_ready_tree(item)
        return
    if isinstance(value, (tuple, list)):
        for item in value:
            block_until_ready_tree(item)
        return
    if hasattr(value, "mantissa"):
        block_until_ready_tree(value.mantissa)


def sync_network_result(result):
    block_until_ready_tree(result.time)
    block_until_ready_tree(result.traces)
    block_until_ready_tree(result.spikes)


def run_network_once(net):
    result = net.run(
        dt=DT,
        duration=DURATION,
        event_backend=EVENT_BACKEND,
        brainevent_backend=BRAINEVENT_BACKEND,
        spike_recording=SPIKE_RECORDING,
    )
    sync_network_result(result)
    return result


In [ ]:
def run_recurrent_case(size: int):
    build_cell_start = time.perf_counter()
    cell = build_pc_population(size)
    build_cell_seconds = time.perf_counter() - build_cell_start

    init_reset_start = time.perf_counter()
    cell.init_state()
    cell.reset_state()
    init_reset_seconds = time.perf_counter() - init_reset_start

    build_network_start = time.perf_counter()
    net = braincell.Network(name=f"pc_recurrent_N{size}")
    net.add_population("PC", cell)
    edges = net.add_edges(
        name="PC_to_PC",
        pre="PC",
        post="PC",
        method=recurrent_edge_method(),
    )
    add_recurrent_projection(net, edges)
    build_network_seconds = time.perf_counter() - build_network_start

    cold_start = time.perf_counter()
    plot_result = run_network_once(net)
    cold_seconds = time.perf_counter() - cold_start

    warm_times = []
    warm_results = []
    for _ in range(WARM_REPEATS):
        cell.reset_state()
        warm_start = time.perf_counter()
        warm_result = run_network_once(net)
        warm_times.append(time.perf_counter() - warm_start)
        warm_results.append(warm_result)

    warm_arr = np.asarray(warm_times, dtype=float)
    warmup_after_jit_seconds = float(warm_arr[0]) if warm_arr.size else float("nan")
    warm_median_seconds = float(np.median(warm_arr)) if warm_arr.size else float("nan")
    warm_min_seconds = float(np.min(warm_arr)) if warm_arr.size else float("nan")
    compile_overhead_seconds = cold_seconds - warm_median_seconds
    compile_overhead_pct = 100.0 * compile_overhead_seconds / cold_seconds if cold_seconds > 0.0 else float("nan")

    steps = int(plot_result.time.shape[0])
    spikes = array_2d(plot_result.spikes["PC"]) if "PC" in plot_result.spikes else np.zeros((steps, 0), dtype=float)
    g_syn = trace_2d(plot_result.traces["PC"]["g_syn"], u.uS)
    summary = {
        "N": int(size),
        "edges": int(edges.n_edge),
        "steps": steps,
        "build_cell_s": build_cell_seconds,
        "init_reset_s": init_reset_seconds,
        "build_network_s": build_network_seconds,
        "cold_run_s": cold_seconds,
        "cold_ms_per_step": 1000.0 * cold_seconds / max(steps, 1),
        "warmup_after_jit_run_s": warmup_after_jit_seconds,
        "warmup_after_jit_ms_per_step": 1000.0 * warmup_after_jit_seconds / max(steps, 1),
        "warm_run_median_s": warm_median_seconds,
        "warm_run_min_s": warm_min_seconds,
        "warm_ms_per_step": 1000.0 * warm_median_seconds / max(steps, 1),
        "compile_plus_first_run_overhead_s": compile_overhead_seconds,
        "compile_plus_first_run_overhead_pct": compile_overhead_pct,
        "spike_recording": SPIKE_RECORDING,
        "spike_shape": tuple(spikes.shape),
        "spike_count": int(spikes.sum()),
        "g_syn_ever_positive": bool(np.any(g_syn > 0.0)),
    }
    return summary, plot_result, edges


summaries = []
case_results = {}
case_edges = {}

for size in N_LIST:
    print(f"running N={size} ...")
    summary, plot_result, edges = run_recurrent_case(size)
    summaries.append(summary)
    case_results[size] = plot_result
    case_edges[size] = edges
    print(summary)


In [ ]:
try:
    import pandas as pd

    summary_table = pd.DataFrame(summaries)
    display(summary_table)
except Exception:
    summary_table = summaries
    for row in summaries:
        print(row)


In [ ]:
if ENABLE_JAX_PROFILE and case_results:
    import jax.profiler

    profile_n = max(case_results)
    profile_cell = build_pc_population(profile_n)
    profile_cell.init_state()
    profile_cell.reset_state()
    profile_net = braincell.Network(name=f"pc_recurrent_profile_N{profile_n}")
    profile_net.add_population("PC", profile_cell)
    profile_edges = profile_net.add_edges(
        name="PC_to_PC",
        pre="PC",
        post="PC",
        method=recurrent_edge_method(),
    )
    add_recurrent_projection(profile_net, profile_edges)

    run_network_once(profile_net)
    profile_cell.reset_state()
    jax.profiler.start_trace(PROFILE_DIR)
    try:
        run_network_once(profile_net)
    finally:
        jax.profiler.stop_trace()
    print("wrote JAX profiler trace to:", PROFILE_DIR)


In [ ]:
if summaries:
    n_values = np.asarray([row["N"] for row in summaries], dtype=float)
    edge_values = np.asarray([row["edges"] for row in summaries], dtype=float)
    cold_ms_per_step = np.asarray([row["cold_ms_per_step"] for row in summaries], dtype=float)
    warmup_ms_per_step = np.asarray([row["warmup_after_jit_ms_per_step"] for row in summaries], dtype=float)
    warm_ms_per_step = np.asarray([row["warm_ms_per_step"] for row in summaries], dtype=float)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(n_values, cold_ms_per_step, marker="o", label="cold")
    axes[0].plot(n_values, warmup_ms_per_step, marker="o", label="first warm")
    axes[0].plot(n_values, warm_ms_per_step, marker="o", label="warm median")
    axes[0].set_xlabel("population size N")
    axes[0].set_ylabel("run wall ms / step")
    axes[0].set_title("Scaling by population size")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(edge_values, cold_ms_per_step, marker="o", label="cold")
    axes[1].plot(edge_values, warmup_ms_per_step, marker="o", label="first warm")
    axes[1].plot(edge_values, warm_ms_per_step, marker="o", label="warm median")
    axes[1].set_xlabel("recurrent edge count")
    axes[1].set_ylabel("run wall ms / step")
    axes[1].set_title("Scaling by recurrent edges")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    fig.suptitle("Formal PC recurrent population benchmark")
    fig.tight_layout()
    plt.show()


In [ ]:
if case_results:
    plot_n = max(case_results)
    result = case_results[plot_n]
    edges = case_edges[plot_n]

    time_ms = np.asarray(result.time.to_decimal(u.ms), dtype=float)
    v_mV = trace_2d(result.traces["PC"]["v"], u.mV)
    g_uS = trace_2d(result.traces["PC"]["g_syn"], u.uS)
    spikes = array_2d(result.spikes["PC"]) if "PC" in result.spikes else np.zeros((time_ms.shape[0], plot_n), dtype=float)

    print("plot case N:", plot_n)
    print("edge pairs:", list(zip(edges.pre_index.tolist(), edges.post_index.tolist())))
    print("spike count by cell:", spikes.sum(axis=0).astype(int))
    print("g_syn ever positive by cell:", np.any(g_uS > 0.0, axis=0))

    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    colors = plt.cm.tab10(np.linspace(0.0, 1.0, max(plot_n, 1)))
    for idx in range(plot_n):
        axes[0].plot(time_ms, v_mV[:, idx], color=colors[idx], linewidth=1.2, label=f"PC[{idx}]")
        axes[1].plot(time_ms, g_uS[:, idx], color=colors[idx], linewidth=1.2, label=f"PC[{idx}]")
        axes[2].step(time_ms, spikes[:, idx], where="post", color=colors[idx], linewidth=1.0, label=f"PC[{idx}]")

    axes[0].set_ylabel("soma v (mV)")
    axes[1].set_ylabel("ExpSyn g (uS)")
    axes[2].set_ylabel("spike")
    axes[2].set_xlabel("time (ms)")
    for axis in axes:
        axis.grid(True, alpha=0.3)
        if plot_n <= 10:
            axis.legend(loc="upper right", ncol=min(4, plot_n), fontsize=8)
    fig.suptitle(f"All PC soma voltage traces, N={plot_n}")
    fig.tight_layout()
    plt.show()
